# Chapter 3 : Routing
# What is Routing ?

# Routing means using an initial Gemini call to decide which path the input should take.

# Doing API Authentication

In [20]:
from google import genai
from google.genai import types

# Developer TODO: Replace YOUR_API_KEY with your API key.
API_KEY = "Enter your API Key"

client = genai.Client(
    vertexai=False, api_key=API_KEY
)

In [21]:
import os

os.environ["GEMINI_API_KEY"] = "Enter your API Key"

In [22]:
chat = client.chats.create(model="gemini-3.5-flash")

In [26]:
# Step 1 — Build a simple router

In [41]:
from google.adk.tools import FunctionTool
from google.adk.agents import Agent
import os

In [42]:
def booking_handler(request : str) -> str:
  """Handles booking request for flights and hotels.
  Args:
    request : The user's request for a booking
    Returns : A confirmation message the time booking was handled
      """
  print("Booking Handler called")
  return f"Booking action for '{request}' has been handled."

In [43]:
def info_handler(request : str) -> str:
  """ Handles general information request
  Args :
    request : The user's question.
  Returns :
   A msg indicating the information request was handled  """
  print("Info Handler called")
  return f"Information request for '{request}'. Result Simulated information retrieval  "


In [44]:
def unclear_handler(request : str) -> str:
  """ Handles request that cannot be delegated """
  print("Unclear Handler called")
  return f"Unclear request for '{request}'. Result Simulated information retrieval"

In [45]:
booking_tool = FunctionTool(booking_handler)
info_tool = FunctionTool(info_handler)

In [46]:
# --define subagents for specific tools ---

booking_agent = Agent(name = "booker",
    model="gemini-3.5-flash",
    description = "A specialized agent for booking flights and hotels",
    tools = [booking_tool])

In [47]:
# --define subagents for specific tools ---

info_agent = Agent(name = "info",
    model="gemini-3.5-flash",
    description = "A specialized agent to answer general queries",
    tools = [info_tool])

In [48]:
# Define Parent Agent with delgation instructions

In [49]:
coordinator = Agent(
    name="Coordinator",

    model="gemini-3.5-flash",

    instruction="""
    You are the main coordinator.
    Your only task is to analyze incoming user requests
    and delegate them to the appropriate specialist agent.
    Do not try to answer directly.

    For any requests related to booking flights or hotels,
    delegate to the 'Booker' agent.

    For all other general information questions,
    delegate to the 'Info_agent'.
    """,

    description="A coordinator that routes user requests to the correct specialist agent",

    sub_agents=[booking_agent, info_agent]
)

In [50]:
# --- Execution Logic --
from google.adk.runners import InMemoryRunner
import uuid

In [51]:
async def run_coordinator(runner, request):

    async for event in runner.run_async(
        user_id="user",
        session_id="session",
        new_message=types.Content(
            role="user",
            parts=[types.Part(text=request)]
        )
    ):
        print(event)

In [52]:
async def run_coordinator(runner, request):

    await runner.session_service.create_session(
        app_name=runner.app_name,
        user_id="user",
        session_id="session"
    )

    async for event in runner.run_async(
        user_id="user",
        session_id="session",
        new_message=types.Content(
            role="user",
            parts=[types.Part(text=request)]
        )
    ):
        if event.is_final_response():
            return event.content.parts[0].text

In [53]:
async def main():

    runner = InMemoryRunner(coordinator)

    result = await run_coordinator(
        runner,
        "Book me a hotel in Paris"
    )

    print(result)

In [54]:
await main()

Booking Handler called
I have processed your booking request for a hotel in Paris.



# --- THE END ---

